# MODEL_COMPARISION — three Qwen candidates / M3-01

**Colab A100 preliminary comparison:** Qwen3-8B BF16, Qwen3.5-9B BF16, Qwen3-14B **official FP8**. Models run sequentially, never together. This compares deployable configurations, not model size alone: precision and architecture differ.

Fill REPO_URL, select a GPU runtime, then run cells in order. Each criterion has an explanatory cell. All results are generated from real calls; this notebook contains no pre-filled scores. No changes to src. Evidence goes outside the repository under /content/datamate_comparison.

Status vocabulary:
- **PASS:** all observed cases satisfy the explicitly stated check.
- **FAIL:** a check ran and a violation was observed; it does not mean every case failed.
- **PARTIAL:** some evidence exists but requested coverage is incomplete.
- **NOT VERIFIED:** the check was not run or sufficient evidence is unavailable.
- **BLOCKED:** execution of later checks stopped because setup/startup/warm-up failed.
A100 results do not certify sm_120, RTX 5090, Docker, full-run PA or the official deadline/runtime gate.

The workload and prompt bytes are shared. Token counts differ by tokenizer. Report numerators AND denominators: 28/29 exceeds 23/29 only for the same cases, metric and conditions; neither passes a strict 29/29 check. A stable wrong answer is still wrong.

Repo references: [TASKS M3-01/M3-08](../docs/TASKS.md), [SPEC §7.13/§18](../docs/SPEC.md), [model strategy](../docs/ARCHITECTURE_PROPOSAL.md).

## 1. Experiment configuration

Qwen3.5 needs a newer serving stack than the original single-model notebook. All three candidates use the **same isolated vLLM 0.17.0** environment; do not compare these timing figures directly to the old v0.10.1.1 run. Transformers is resolved within vLLM's declared constraints, then frozen once for all three candidates.

The 14B checkpoint is FP8 to fit the 32 GiB-equivalent budget. On A100, FP8 may use weight-only W8A16/Marlin rather than native FP8 compute; performance is hardware-specific. No automatic precision changes or budget increases are made after failure.
Sources: [supported models](https://docs.vllm.ai/en/v0.17.0/models/supported_models/), [FP8 on Ampere](https://docs.vllm.ai/en/v0.17.0/features/quantization/fp8/).

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, subprocess, hashlib, time, csv, math, statistics
from concurrent.futures import ThreadPoolExecutor, as_completed


REPO_URL = "https://github.com/Kyoyooo/DataMate.git"  # REQUIRED. Do not embed credentials.
REPO_REF = ""  # Prefer an immutable commit; empty uses the default checkout.
REPO_DIR = Path("/content/DataMate")
EVAL_ROOT = Path("/content/datamate_comparison")
VENV_DIR = Path("/content/datamate_comparison_py312")
VLLM_VERSION = "0.17.0"
PORT = 8000
BASE_URL = f"http://127.0.0.1:{PORT}"
SERVED_NAME = "datamate-comparison"
SEED = 20261116
MAX_MODEL_LEN = 8192
MAX_NUM_SEQS = 8
TARGET_EXECUTOR_GIB = 28.8
CONCURRENCIES = [1, 4]
REPEATS = 2
REQUEST_TIMEOUT_S = 180
STARTUP_TIMEOUT_S = 1200
EXECUTE_FIXTURE_SQL = True
SQL_TIMEOUT_S = 2.0
WEIGHT_CUTOFF = datetime.fromisoformat("2026-10-26T00:00:00+07:00")
HASH_WEIGHTS = True  # SHA-256 all checkpoint shards; may take several minutes.
MODELS = [
    dict(key="qwen3_8b", model_id="Qwen/Qwen3-8B", revision="",
         dtype="bfloat16", quantization=None, base_model_id=None, base_revision="",
         extra_args=[], license_reviewed=False, release_evidence_url=""),
    dict(key="qwen35_9b", model_id="Qwen/Qwen3.5-9B", revision="",
         dtype="bfloat16", quantization=None, base_model_id=None, base_revision="",
         extra_args=["--language-model-only"], license_reviewed=False, release_evidence_url=""),
    dict(key="qwen3_14b_fp8", model_id="Qwen/Qwen3-14B-FP8", revision="",
         dtype="bfloat16", quantization="fp8", base_model_id="Qwen/Qwen3-14B", base_revision="",
         extra_args=[], license_reviewed=False, release_evidence_url=""),
]
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR = EVAL_ROOT / RUN_STAMP
RUN_DIR.mkdir(parents=True, exist_ok=False)
assert REPEATS >= 2 and max(CONCURRENCIES) <= MAX_NUM_SEQS

def save_json(name, value):
    path = RUN_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8")
    return path

def run_command(args, **kwargs):
    return subprocess.run([str(x) for x in args], check=True, text=True, **kwargs)

print("Run:", RUN_DIR)

Run: /content/datamate_comparison/20261109T111420_557266Z


## 2. Clone and freeze the project

A clean commit fixes the same contracts and prompts for every candidate. PASS here means a readable clean checkout, not that the agent is correct. Missing URL, dirty checkout or absent fixture stops setup. Authentication for private repos must be configured separately; do not store tokens in this notebook.

In [2]:
if not REPO_URL.strip():
    raise ValueError("Fill REPO_URL in section 1, then rerun from here.")
if not REPO_DIR.exists():
    run_command(["git", "clone", "--", REPO_URL, REPO_DIR])
elif not (REPO_DIR / ".git").is_dir():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout.")
if REPO_REF:
    run_command(["git", "-C", REPO_DIR, "checkout", "--detach", REPO_REF])
dirty = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True
).strip()
if dirty:
    raise RuntimeError("Checkout has local changes. Use a clean checkout or a separate REPO_DIR.")
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
for relative in ["src/datamate/contracts/types.py", "src/datamate/prompts/templates.py",
                 "tests/fixtures/split_mini/questions.jsonl"]:
    assert (REPO_DIR / relative).is_file(), relative
print("Repository commit:", REPO_COMMIT)

Repository commit: d24001107dec9a3c431f987d55a15f09822420d5


## 3. GPU identity

Record name, UUID, driver, capacity and baseline allocation. No GPU means setup FAIL. A100 identity is not a requirement to change into sm_120: target-hardware compatibility remains NOT VERIFIED. Compare performance only when GPU identity, software and settings match.

In [3]:
GPU_QUERY = ["nvidia-smi", "--query-gpu=index,name,uuid,memory.total,memory.used,driver_version",
             "--format=csv,noheader,nounits"]

def gpu_snapshot():
    raw = subprocess.check_output(GPU_QUERY, text=True)
    result = []
    for row in csv.reader(raw.splitlines()):
        index, name, uuid, total, used, driver = [x.strip() for x in row]
        result.append(dict(index=int(index), name=name, uuid=uuid,
                           total_mib=float(total), used_mib=float(used), driver=driver))
    return result

GPUS = gpu_snapshot()
assert GPUS, "No NVIDIA GPU. Select an A100 GPU runtime."
print(json.dumps(GPUS, indent=2))
print("Notebook Python:", sys.version)
if "A100" not in GPUS[0]["name"]:
    print("NOTICE: This is not an A100; retain the actual GPU identity in the report.")
save_json("gpu_before_setup.json", GPUS)

[
  {
    "index": 0,
    "name": "NVIDIA L4",
    "uuid": "GPU-166c58ef-65ed-9d23-d7ca-62e2cc9d89c5",
    "total_mib": 23034.0,
    "used_mib": 0.0,
    "driver": "580.82.07"
  }
]
Notebook Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
NOTICE: This is not an A100; retain the actual GPU identity in the report.


PosixPath('/content/datamate_comparison/20261109T111420_557266Z/gpu_before_setup.json')

## 4. Shared isolated serving environment

The notebook kernel imports only CPU evaluation dependencies. A separate Python 3.12 environment owns vLLM/PyTorch; no model is loaded into the kernel. Installation failure blocks all candidates; preserve the error rather than upgrading silently. The resolved serving lock and CUDA/GPU fingerprint apply to all three models.

In [4]:
run_command([sys.executable, "-m", "pip", "install", "--quiet",
             "uv==0.8.15", "duckdb==1.3.2", "pydantic==2.11.7",
             "requests==2.32.5", "huggingface-hub==0.34.4", "jsonschema==4.25.1"])
import shutil
UV = shutil.which("uv")
assert UV, "uv executable not found after installation."
if not VENV_DIR.exists():
    run_command([UV, "venv", "--python", "3.12", "--seed", VENV_DIR])
SERVER_PYTHON = VENV_DIR / "bin/python"
run_command([UV, "pip", "install", "--python", SERVER_PYTHON,
             f"vllm=={VLLM_VERSION}"])
run_command([SERVER_PYTHON, "-m", "pip", "check"])
sys.path.insert(0, str(REPO_DIR / "src"))
import requests
from huggingface_hub import HfApi, snapshot_download
from IPython.display import display, Markdown

kernel_freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
server_freeze = subprocess.check_output([str(SERVER_PYTHON), "-m", "pip", "freeze"], text=True)
(RUN_DIR / "kernel_requirements.txt").write_text(kernel_freeze, encoding="utf-8")
(RUN_DIR / "serving_requirements.txt").write_text(server_freeze, encoding="utf-8")
server_probe = subprocess.check_output([str(SERVER_PYTHON), "-c",
    "import torch,json,sys,vllm,transformers; print(json.dumps(dict("
    "python=sys.version,torch=torch.__version__,cuda=torch.version.cuda,"
    "vllm=vllm.__version__,transformers=transformers.__version__,"
    "gpu=torch.cuda.get_device_name(0),capability=torch.cuda.get_device_capability(0))))"
], text=True)
SERVER_ENV = json.loads(server_probe.strip().splitlines()[-1])
save_json("serving_environment.json", SERVER_ENV)
print(json.dumps(SERVER_ENV, indent=2))

{
  "python": "3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]",
  "torch": "2.10.0+cu128",
  "cuda": "12.8",
  "vllm": "0.17.0",
  "transformers": "4.57.6",
  "gpu": "NVIDIA L4",
  "capability": [
    8,
    9
  ]
}


## 5. Identical fixture workload and reference contexts

Reuse the repository's original prompts, schemas, formatter, scorer and read-only executor. Gold frames from the scripted fixture isolate the solver from parser errors; SQL responses are generated by the real model. No public/private data is used.

Cases comprise fixture parse/solve, temporal diagnostics, one repair and one fixed long-context probe. The long probe has identical bytes across models, unlike model-specific padding. It is excluded from answer accuracy. The source executor has timer-based interruption, not process isolation.

In [5]:
from datamate.contracts import CONTRACTS_VERSION, Question, QuestionFrame, SolverOutput
from datamate.contracts import Decision, SolverAttempt
from datamate.config import RunConfig, ExecLimits
from datamate.catalog import setup_business
from datamate.rules import load_split
from datamate.temporal import resolve_windows
from datamate.prompts import render_parse, render_solve, render_repair, template_sha256
from datamate.lexicon import fold
from datamate.decision import pre_solve_decision
from datamate.solver import check_sql, to_typed_value
from datamate.sandbox import run_sql
from datamate.formatting import to_submission
from datamate.eval.score import score_one
from datetime import date

SPLIT_DIR = REPO_DIR / "tests/fixtures/split_mini"
split = load_split(SPLIT_DIR)
questions_raw = [json.loads(x) for x in (SPLIT_DIR / "questions.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
references = {x["id"]: x for x in [json.loads(s) for s in
    (SPLIT_DIR / "answers.jsonl").read_text(encoding="utf-8").splitlines() if s.strip()]}
script_entries = [json.loads(s) for s in (REPO_DIR / "tests/fixtures/scripted/split_mini.jsonl").read_text(
    encoding="utf-8").splitlines() if s.strip()]
reference_frames = {x["question"]: QuestionFrame.model_validate_json(x["content"])
                    for x in script_entries if x["stage"] == "parse"}
catalogs = {}
cases = []
cfg = RunConfig()
for line_no, raw in enumerate(questions_raw, 1):
    q = Question(**{**raw, "as_of_date": date.fromisoformat(raw["as_of_date"]), "line_no": line_no})
    if q.business_id not in catalogs:
        catalogs[q.business_id] = setup_business(
            SPLIT_DIR / "data" / q.business_id, q.business_id, RUN_DIR / "work")
    cat = catalogs[q.business_id]
    cases.append(dict(case_id=f"parse:{q.id}", stage="parse", template_id="parse.frame.v3",
        messages=render_parse(q, split, cat), schema=QuestionFrame.model_json_schema(),
        max_tokens=1024, question=q, reference=references[q.id], diagnostic=False))
    frame = reference_frames[q.question]
    windows, error = resolve_windows(frame.time, q.as_of_date, split.temporal, cat.data_range)
    if not error and pre_solve_decision(frame, windows, cat, cfg) is None:
        cases.append(dict(case_id=f"solve:{q.id}", stage="solve", template_id="solve.sql.v1",
            messages=render_solve(q, frame, windows, split, cat), schema=SolverOutput.model_json_schema(),
            max_tokens=2048, question=q, frame=frame, reference=references[q.id], diagnostic=False))

cat = next(iter(catalogs.values()))
temporal_texts = [
    "Từ 1/3 đến 4/3 có nhiều đơn hơn cùng kỳ tháng trước không?",
    "Doanh thu 2 tháng gần nhất là bao nhiêu?",
    "Doanh thu tháng này so với cùng kỳ năm ngoái thay đổi bao nhiêu phần trăm?",
]
for i, text in enumerate(temporal_texts):
    q = Question(id=f"temporal_{i}", business_id=cat.business_id, question=text,
                 as_of_date=date(2026, 3, 4), line_no=100 + i)
    cases.append(dict(case_id=f"parse:temporal_{i}", stage="parse", template_id="parse.frame.v3",
        messages=render_parse(q, split, cat), schema=QuestionFrame.model_json_schema(),
        max_tokens=1024, question=q, diagnostic=True))
first_solve = next(c for c in cases if c["stage"] == "solve")
bad_sql = json.dumps({"sql": "SELECT deliberately_missing_column FROM ban_hang", "notes": ""})
cases.append({**first_solve, "case_id": "repair:invalid_column", "stage": "repair",
    "template_id": "repair.sql.v1", "messages": render_repair(first_solve["messages"], bad_sql,
    "E_SQL_EXEC", "Referenced column deliberately_missing_column was not found.")})
print("Fixture cases:", len(cases), "| contracts:", CONTRACTS_VERSION)
print("Templates:", template_sha256())

import copy
long_case = copy.deepcopy(first_solve)
long_case.update(case_id="solve:long_context_diagnostic", diagnostic=True)
long_case.pop("reference", None)
padding = "\nNOTE: Synthetic catalog context padding for serving tests only." * 200
long_case["messages"][0]["content"] = long_case["messages"][0]["content"].replace(
    "\nDATA SCHEMA:", padding + "\nDATA SCHEMA:", 1)
cases.append(long_case)
CASE_IDS = [c["case_id"] for c in cases]
assert len(CASE_IDS) == len(set(CASE_IDS))
parse_cases = [c for c in cases if c["stage"] == "parse"]
answer_cases = [c for c in cases if c["stage"] != "parse" and not c["diagnostic"] and "reference" in c]
expected_requests = len(cases) * len(CONCURRENCIES) * REPEATS
workload_payload = [{k:c[k] for k in ("case_id","stage","messages","schema","max_tokens","diagnostic")} for c in cases]
WORKLOAD_SHA256 = hashlib.sha256(json.dumps(workload_payload, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
save_json("workload.json", workload_payload)
print("Shared cases:", len(cases), "| requests per model:", expected_requests)
print("Workload SHA-256:", WORKLOAD_SHA256)

Fixture cases: 28 | contracts: 1.3.0
Templates: {'parse.frame.v3': '83fd39fe54c0b51eca2c1b63946fcfc7094957d2e6f26975d3965abbda03f359', 'repair.sql.v1': '369c4671f61c8fa538895b4d0167c6471837aecc032a689026c621f905a50051', 'rules.temporal.v1': '2fa07419a6e62d9b5e57e7d14495d0dcf28ca4a3e1d261c640cf2ac8f5484273', 'solve.sql.v1': 'ca4c1801325b71349a6aebf0834f0247a519f149a198f7234c571c1ab741372e'}
Shared cases: 29 | requests per model: 116
Workload SHA-256: 286becec3b90366a956f12781080dd3dbd4ddc5aaae97396b8a3d5dbc34afc66


## 6. Schema compatibility preflight

This carries forward the oneOf fix: only TimeRef.expr is copied to anyOf, with required, distinct kind.const tags. Arbitrary oneOf unions are rejected. Original Pydantic validation remains authoritative. PASS means the adapter's preflight succeeded; backend compilation is checked later by warm-up. Both schemas and their hashes are saved.

In [6]:
import copy
from jsonschema import Draft202012Validator

SCHEMA_ADAPTER_POLICY = "timeref-kind-required-oneof-to-anyof.v1"

def schema_sha256(schema):
    canonical = json.dumps(schema, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

def schema_for_vllm(original):
    """Adapt only TimeRef.expr; application validation still uses the original model."""
    wire = copy.deepcopy(original)
    definitions = wire.get("$defs", {})
    expr = definitions.get("TimeRef", {}).get("properties", {}).get("expr", {})
    if "oneOf" in expr:
        if expr.get("discriminator", {}).get("propertyName") != "kind":
            raise ValueError("Unexpected TimeRef discriminator; review the schema adapter.")
        branches = expr["oneOf"]
        if not isinstance(branches, list) or not branches:
            raise ValueError("TimeRef.expr.oneOf must be a non-empty list.")
        seen_kinds = set()
        for branch in branches:
            ref = branch.get("$ref", "")
            if set(branch) != {"$ref"} or not ref.startswith("#/$defs/"):
                raise ValueError(f"Unsupported TimeRef branch: {branch}")
            definition = definitions[ref.removeprefix("#/$defs/")]
            kind = definition.get("properties", {}).get("kind", {})
            tag = kind.get("const")
            if definition.get("type") != "object" or kind.get("type") != "string":
                raise ValueError(f"Expected an object with a string kind: {ref}")
            if not isinstance(tag, str) or tag in seen_kinds:
                raise ValueError(f"Missing or overlapping kind.const: {ref}")
            seen_kinds.add(tag)
            # Pydantic's tagged union needs kind even when a variant has a default.
            required = definition.setdefault("required", [])
            if "kind" not in required:
                required.append("kind")
        # Required, distinct kind.const values make the branches mutually exclusive.
        expr["anyOf"] = expr.pop("oneOf")
        expr.pop("discriminator", None)

    def reject_remaining_oneof(node, location="$"):
        # Walk schema-valued keywords only; a property named 'oneOf' is not a keyword.
        if isinstance(node, dict):
            if "oneOf" in node:
                raise ValueError(f"Unsupported oneOf remains at {location}; review explicitly.")
            for key in ("$defs", "definitions", "properties", "patternProperties", "dependentSchemas"):
                for name, child in node.get(key, {}).items():
                    reject_remaining_oneof(child, f"{location}/{key}/{name}")
            for key in ("items", "additionalProperties", "contains", "not", "if", "then", "else", "propertyNames"):
                child = node.get(key)
                if isinstance(child, dict):
                    reject_remaining_oneof(child, f"{location}/{key}")
            for key in ("anyOf", "allOf", "prefixItems"):
                for index, child in enumerate(node.get(key, [])):
                    reject_remaining_oneof(child, f"{location}/{key}/{index}")
    reject_remaining_oneof(wire)
    Draft202012Validator.check_schema(wire)
    return wire

schema_evidence = {"policy": SCHEMA_ADAPTER_POLICY, "templates": {}}
for case in cases:
    original = case["schema"]
    serving = schema_for_vllm(original)
    case["serving_schema"] = serving
    evidence = {
        "original_sha256": schema_sha256(original),
        "serving_sha256": schema_sha256(serving),
        "changed": original != serving,
        "original_schema": original,
        "serving_schema": serving,
    }
    previous = schema_evidence["templates"].get(case["template_id"])
    if previous is not None and previous != evidence:
        raise ValueError("A template ID has conflicting schemas in this run.")
    schema_evidence["templates"][case["template_id"]] = evidence
save_json("schema_evidence.json", schema_evidence)
for template_id, evidence in schema_evidence["templates"].items():
    print(template_id, "adapted:", evidence["changed"],
          "serving schema SHA-256:", evidence["serving_sha256"])

parse.frame.v3 adapted: True serving schema SHA-256: a42a3bd15e00791dd88e9246792a94401d8dcf371cef61984f93a8e2dd76f0d8
solve.sql.v1 adapted: False serving schema SHA-256: 907e621e5a6b0d9b9048b9c023944ad2a3da2c8d73ccfc1a8ee9a7f0d2387889
repair.sql.v1 adapted: False serving schema SHA-256: 907e621e5a6b0d9b9048b9c023944ad2a3da2c8d73ccfc1a8ee9a7f0d2387889


## 7. Checkpoint eligibility and resource preparation helpers

Each model is resolved to an immutable pre-cutoff revision, downloaded locally and hashed. Original weights release/license evidence still needs human review. A model name is never a parameter-count proof.

For BF16 checkpoints, count actual safetensors tensor shapes including shipped vision/MTP tensors. For FP8, report stored elements separately and obtain full base-model parameter metadata at a pinned revision; quantization scales must not be mistaken for learned parameters. Missing metadata is NOT VERIFIED, not a guessed PASS.

Download/startup failures are preserved per model; other candidates still run. Keep about 60+ GiB free for checkpoints plus serving dependencies/caches. Downloads occur during preparation, never inside benchmark requests.

In [7]:
import struct, traceback, shutil, signal, threading
from contextlib import contextmanager
api = HfApi()
MODEL_RUNS = {}

def resolve_revision(model_id, requested=""):
    commits = list(api.list_repo_commits(model_id))
    eligible = [c for c in commits if c.created_at < WEIGHT_CUTOFF]
    if requested:
        revision = api.model_info(model_id, revision=requested).sha
        commit = next((c for c in commits if c.commit_id == revision), None)
        if commit is None or commit.created_at >= WEIGHT_CUTOFF:
            raise ValueError("Requested checkpoint revision is not verified before cutoff.")
    else:
        if not eligible:
            raise ValueError("No checkpoint commit before cutoff.")
        commit = max(eligible, key=lambda c:c.created_at)
        revision = commit.commit_id
    return revision, commit.created_at.isoformat()

def prepare_checkpoint(profile, state):
    model_id = profile["model_id"]
    revision, revision_date = resolve_revision(model_id, profile["revision"])
    info = api.model_info(model_id, revision=revision)
    path = Path(snapshot_download(repo_id=model_id, revision=revision,
        allow_patterns=["*.safetensors","*.json","*.txt","*.model","*.tiktoken",
                        "*.jinja","*.jinja2","README.md","LICENSE*"]))
    shards = sorted(path.glob("*.safetensors"))
    if not shards:
        raise RuntimeError("No safetensors checkpoint found.")
    elements, dtypes, hashes = 0, set(), {}
    for shard in shards:
        with shard.open("rb") as fh:
            length = struct.unpack("<Q",fh.read(8))[0]
            if length > 100_000_000:
                raise ValueError("Unexpected safetensors header size.")
            header = json.loads(fh.read(length))
        for name,tensor in header.items():
            if name != "__metadata__":
                elements += math.prod(tensor["shape"])
                dtypes.add(tensor["dtype"])
        if HASH_WEIGHTS:
            digest = hashlib.sha256()
            with shard.open("rb") as fh:
                for block in iter(lambda:fh.read(16*1024*1024),b""):
                    digest.update(block)
            hashes[shard.name]=digest.hexdigest()
    count, count_source, base_evidence = None, None, None
    if not profile["quantization"] and dtypes <= {"BF16","F16","F32"}:
        count, count_source = elements, "actual unquantized safetensors tensor shapes"
    elif profile["base_model_id"]:
        base_rev, base_date = resolve_revision(profile["base_model_id"],profile["base_revision"])
        base_info = api.model_info(profile["base_model_id"],revision=base_rev)
        safetensors = getattr(base_info,"safetensors",None)
        count = getattr(safetensors,"total",None)
        if isinstance(safetensors,dict):
            count = safetensors.get("total")
        count_source = "Hub metadata of pinned unquantized base; review before release"
        base_evidence = dict(model_id=profile["base_model_id"],revision=base_rev,date=base_date)
    card = info.card_data.to_dict() if info.card_data else {}
    evidence = dict(model_id=model_id,revision=revision,revision_date=revision_date,
        license=card.get("license"),license_reviewed=profile["license_reviewed"],
        release_evidence_url=profile["release_evidence_url"],stored_tensor_elements=elements,
        parameter_count=count,parameter_count_source=count_source,base_evidence=base_evidence,
        dtypes=sorted(dtypes),weights_sha256=hashes,weight_hashes_complete=HASH_WEIGHTS)
    state.update(checkpoint=evidence,model_path=str(path))
    save_json(f'{profile["key"]}/checkpoint.json',evidence)
    if count is not None and count > 15_000_000_000:
        raise ValueError("Parameter count exceeds 15B; candidate not served.")
    return path

TOKENIZER_PROGRAM = """
import sys,json
from transformers import AutoTokenizer
t=AutoTokenizer.from_pretrained(sys.argv[1],local_files_only=True)
messages=json.load(sys.stdin)
print(json.dumps([len(t.apply_chat_template(m,tokenize=True,
    add_generation_prompt=True,enable_thinking=False)) for m in messages]))
"""
def count_candidate_tokens(path):
    output = subprocess.check_output([str(SERVER_PYTHON),"-c",TOKENIZER_PROGRAM,str(path)],
        input=json.dumps([c["messages"] for c in cases],ensure_ascii=False),text=True)
    return json.loads(output.strip().splitlines()[-1])

@contextmanager
def serving_candidate(profile,state,path):
    key=profile["key"]
    with requests.Session() as session:
        session.trust_env=False
        try:
            session.get(BASE_URL+"/health",timeout=2)
        except requests.RequestException:
            pass
        else:
            raise RuntimeError("Port already occupied. Stop the previous server or change PORT.")
    budget_fraction=min(0.90,TARGET_EXECUTOR_GIB*1024/GPUS[0]["total_mib"])
    state["engine_memory_fraction"]=budget_fraction
    args=[str(SERVER_PYTHON),"-m","vllm.entrypoints.openai.api_server",
        "--model",str(path),"--tokenizer",str(path),"--served-model-name",SERVED_NAME,
        "--host","127.0.0.1","--port",str(PORT),"--dtype",profile["dtype"],
        "--max-model-len",str(MAX_MODEL_LEN),"--max-num-seqs",str(MAX_NUM_SEQS),
        "--tensor-parallel-size","1","--gpu-memory-utilization",str(budget_fraction),
        "--seed",str(SEED),"--generation-config","vllm","--reasoning-parser","qwen3",
        "--enable-prefix-caching","--cpu-offload-gb","0",*profile["extra_args"]]
    if profile["quantization"]:
        args += ["--quantization",profile["quantization"]]
    save_json(f"{key}/server_command.json",args)
    env={**os.environ,"CUDA_VISIBLE_DEVICES":"0","HF_HUB_OFFLINE":"1",
         "TRANSFORMERS_OFFLINE":"1","HF_HUB_DISABLE_TELEMETRY":"1",
         "VLLM_NO_USAGE_STATS":"1","TOKENIZERS_PARALLELISM":"false"}
    state["gpu_samples"]=[]
    stop=threading.Event()
    def monitor():
        while not stop.is_set():
            try:
                state["gpu_samples"].append(dict(time=time.time(),**gpu_snapshot()[0]))
            except Exception as exc:
                state["gpu_samples"].append(dict(time=time.time(),error=str(exc)))
            stop.wait(0.5)
    thread=threading.Thread(target=monitor,daemon=True)
    thread.start()
    log=(RUN_DIR/key/"server.log").open("w",encoding="utf-8")
    server=None
    state["startup_attempted"]=True
    started=time.perf_counter()
    try:
        server=subprocess.Popen(args,env=env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
        with requests.Session() as session:
            session.trust_env=False
            while True:
                if server.poll() is not None:
                    raise RuntimeError("Server exited. Inspect this candidate's server.log.")
                try:
                    if session.get(BASE_URL+"/health",timeout=2).status_code==200:
                        break
                except requests.RequestException:
                    pass
                if time.perf_counter()-started>STARTUP_TIMEOUT_S:
                    raise TimeoutError("Startup timeout.")
                time.sleep(2)
        state.update(startup_ok=True,load_seconds=time.perf_counter()-started)
        yield
    finally:
        if server is not None and server.poll() is None:
            os.killpg(server.pid,signal.SIGTERM)
            try:
                server.wait(timeout=30)
            except subprocess.TimeoutExpired:
                os.killpg(server.pid,signal.SIGKILL)
                server.wait(timeout=10)
        stop.set()
        thread.join(timeout=3)
        log.close()
        save_json(f"{key}/gpu_samples.json",state["gpu_samples"])
        # Wait for CUDA allocations owned by the stopped process to be released.
        time.sleep(3)

## 8. Request evaluation and scoring definitions

Each request retains exact HTTP text, schema hashes, content/reasoning and errors. A request is complete only with HTTP/content success, original-model validity and finish_reason=stop. No retry hides failures.

SQL checks and execution/type checks are separate from final answer correctness. Gold-frame solver scores are synthetic fixture results, not full-pipeline PA. Parsed-frame keys fill Pydantic defaults, so missing_fields omitted vs [] can compare equally. SQL keys ignore notes, outer whitespace and trailing semicolons only; they are not proofs of SQL semantic equivalence.

The time-presence diagnostic catches a parser omitting all time expressions for a fixture with an explicit reference time; it does not prove the remaining time expression is correct.

In [8]:
from datamate.contracts import dumps as exact_json_dumps
def request_body(case):
    return {
        "model": SERVED_NAME, "messages": case["messages"],
        "temperature": 0.0, "top_p": 1.0, "top_k": -1, "min_p": 0.0,
        "presence_penalty": 0.0, "frequency_penalty": 0.0, "repetition_penalty": 1.0,
        "seed": SEED, "n": 1, "max_tokens": case["max_tokens"],
        "stream": False, "chat_template_kwargs": {"enable_thinking": False},
        "response_format": {"type": "json_schema", "json_schema": {
            "name": case["template_id"].replace(".", "_"),
            "schema": case["serving_schema"], "strict": True}},
    }

def evaluate_response(case, body, status, raw_text, elapsed, error=None):
    record = dict(case_id=case["case_id"], stage=case["stage"], diagnostic=case["diagnostic"],
        request_sha256=hashlib.sha256(json.dumps(body, sort_keys=True, ensure_ascii=False,
            separators=(",", ":")).encode()).hexdigest(),
        request=body, schema_adapter=SCHEMA_ADAPTER_POLICY,
        original_schema_sha256=schema_sha256(case["schema"]),
        serving_schema_sha256=schema_sha256(case["serving_schema"]),
        http_status=status, raw_response_text=raw_text,
        elapsed_s=elapsed, error=error, transport_ok=False, schema_valid=False,
        app_valid=False, sql_ok=None, typed_ok=None, fixture_score=None,
        content=None, reasoning=None, finish_reason=None,
        prompt_tokens=None, completion_tokens=None)
    if error is not None or status != 200:
        return record
    try:
        payload = json.loads(raw_text)
        choice = payload["choices"][0]
        message = choice["message"]
        record.update(content=message.get("content"),
            reasoning=message.get("reasoning_content", message.get("reasoning")),
            finish_reason=choice.get("finish_reason"),
            prompt_tokens=payload.get("usage", {}).get("prompt_tokens"),
            completion_tokens=payload.get("usage", {}).get("completion_tokens"))
        record["transport_ok"] = isinstance(record["content"], str)
        model = QuestionFrame if case["stage"] == "parse" else SolverOutput
        parsed = model.model_validate_json(record["content"])
        record["schema_valid"] = True
        record["schema_key"] = json.dumps(parsed.model_dump(mode="json"), sort_keys=True, ensure_ascii=False)
        record["app_valid"] = True
        if case["stage"] == "parse":
            record["frame_key"] = record["schema_key"]
            reference_frame = reference_frames.get(case["question"].question)
            if reference_frame is not None:
                record["time_presence_ok"] = not reference_frame.time or bool(parsed.time)
            record["app_valid"] = all(fold(ref.span) in fold(case["question"].question)
                                      for ref in parsed.time)
            if "reference" in case and case["reference"]["type"] not in ("unanswerable", "abstain"):
                record["expected_type_correct"] = parsed.expected_type == case["reference"]["type"]
        else:
            record["sql"] = parsed.sql
            record["sql_key"] = parsed.sql.strip().rstrip(";").rstrip()
            sql_error, sql_detail = check_sql(parsed.sql)
            record.update(sql_ok=sql_error is None, sql_error=sql_error, sql_detail=sql_detail)
            if EXECUTE_FIXTURE_SQL and sql_error is None:
                cat = catalogs[case["question"].business_id]
                execution = run_sql(cat.db_path, parsed.sql, ExecLimits(
                    sql_timeout_s=SQL_TIMEOUT_S, max_result_rows=1000, memory_limit_mb=2048))
                record["execution"] = execution.model_dump(mode="json")
                value, type_error = to_typed_value(case["frame"], execution) if execution.ok else (None, execution.error_code)
                record.update(typed_ok=execution.ok and type_error is None, type_error=type_error)
                if value is not None and "reference" in case and not case["diagnostic"]:
                    decision = Decision(kind="answer", value=value, reason_code="D_ANSWER")
                    submission = to_submission(case["question"].id, decision,
                                               case["frame"].ranking, cfg.formatting)
                    record["fixture_score"] = score_one(submission.to_wire(), case["reference"])
                    record["answer_key"] = exact_json_dumps(submission.to_wire())
    except Exception as exc:
        record["validation_error"] = f"{type(exc).__name__}: {exc}"
    return record

def call_case(case):
    body = request_body(case)
    started = time.perf_counter()
    try:
        with requests.Session() as session:
            session.trust_env = False
            response = session.post(BASE_URL + "/v1/chat/completions", json=body,
                                    timeout=REQUEST_TIMEOUT_S)
        elapsed = time.perf_counter() - started
        return evaluate_response(case, body, response.status_code, response.text, elapsed)
    except requests.RequestException as exc:
        return evaluate_response(case, body, None, None, time.perf_counter() - started,
                                 f"{type(exc).__name__}: {exc}")

## 9. Sequential candidate runner

Each candidate follows the same checkpoint → context-fit → health → schema warm-up → two benchmark passes workflow. A failed model is BLOCKED for unmeasured criteria; it does not receive fabricated zero-accuracy scores. A successful warm-up is not a guarantee all workload requests will succeed.

Request completion order is recorded; input submission order is fixed. Dynamic batching can still vary. Run each candidate cell once; a new experiment requires a new RUN_DIR. Every server is stopped in finally before the next model starts. Downloading three checkpoints may be expensive; results persist independently after each candidate.

In [9]:
def completed(record):
    return (record.get("transport_ok",False) and record.get("schema_valid",False)
            and record.get("finish_reason")=="stop")

def run_candidate(profile):
    key=profile["key"]
    if key in MODEL_RUNS or (RUN_DIR/key).exists():
        raise RuntimeError("Candidate already attempted in this run. Start a new experiment to rerun.")
    (RUN_DIR/key).mkdir()
    state=dict(key=key,profile=profile,status="PREPARING",records=[],batches=[],
        startup_attempted=False,startup_ok=False,warmup_ok=False,
        gpu_uuid=GPUS[0]["uuid"],workload_sha256=WORKLOAD_SHA256)
    MODEL_RUNS[key]=state
    try:
        path=prepare_checkpoint(profile,state)
        counts=count_candidate_tokens(path)
        state["token_counts"]=dict(zip(CASE_IDS,counts,strict=True))
        state["context_fit"]=all(n+c["max_tokens"]<=MAX_MODEL_LEN
                                 for c,n in zip(cases,counts,strict=True))
        save_json(f"{key}/context_counts.json",state["token_counts"])
        if not state["context_fit"]:
            raise ValueError("A shared case exceeds context+output budget; no silent truncation.")
        with serving_candidate(profile,state,path):
            state["status"]="WARMUP"
            state["warmups"]=[call_case(next(c for c in cases if c["stage"]==stage))
                              for stage in ("parse","solve","repair")]
            save_json(f"{key}/warmups.json",state["warmups"])
            state["warmup_ok"]=all(completed(r) for r in state["warmups"])
            if not state["warmup_ok"]:
                for r in state["warmups"]:
                    if not completed(r):
                        print(key,r["stage"],r["http_status"],r.get("error"),
                              r.get("validation_error"),(r.get("raw_response_text") or "")[:1000])
                raise RuntimeError("Warm-up failed. Inspect warmups and schema evidence.")
            state["status"]="BENCHMARKING"
            raw_path=RUN_DIR/key/"requests.jsonl"
            for concurrency in CONCURRENCIES:
                for repeat in range(REPEATS):
                    ordered=[None]*len(cases)
                    start=time.perf_counter()
                    with ThreadPoolExecutor(max_workers=concurrency) as pool:
                        futures={pool.submit(call_case,case):i for i,case in enumerate(cases)}
                        for future in as_completed(futures):
                            r=future.result()
                            r.update(candidate=key,concurrency=concurrency,repeat=repeat)
                            ordered[futures[future]]=r
                            state["records"].append(r)
                            with raw_path.open("a",encoding="utf-8") as fh:
                                fh.write(json.dumps(r,ensure_ascii=False,default=str)+"\n")
                    elapsed=time.perf_counter()-start
                    done=sum(completed(r) for r in ordered)
                    tokens=sum(r.get("completion_tokens") or 0 for r in ordered)
                    state["batches"].append(dict(concurrency=concurrency,repeat=repeat,
                        wall_seconds=elapsed,completed=done,total=len(cases),
                        requests_per_second=done/elapsed,
                        aggregate_output_tokens_per_second=tokens/elapsed))
                    save_json(f"{key}/batches.json",state["batches"])
                    print(key,"concurrency",concurrency,"pass",repeat+1,
                          f"complete {done}/{len(cases)}","seconds",round(elapsed,2))
            state["status"]="COMPLETED"
    except Exception as exc:
        state.update(status="BLOCKED",failure=f"{type(exc).__name__}: {exc}",
                     traceback=traceback.format_exc())
        print(key,"BLOCKED:",state["failure"])
    finally:
        save_json(f"{key}/state.json",state)
    return state["status"]

### Candidate 1: Qwen3-8B BF16

Run the cell below once. COMPLETED means the experiment loop ended, not that all checks passed. BLOCKED means later measurements are NOT VERIFIED; inspect the candidate's state.json, server.log and warmups.json. Proceed to the next candidate even if this one is blocked.

In [10]:
print(run_candidate(MODELS[0]))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

LICENSE: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (7 outputs left out)

model-00005-of-00005.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

qwen3_8b concurrency 1 pass 1 complete 29/29 seconds 189.44
qwen3_8b concurrency 1 pass 2 complete 29/29 seconds 188.21
qwen3_8b concurrency 4 pass 1 complete 29/29 seconds 54.3
qwen3_8b concurrency 4 pass 2 complete 29/29 seconds 54.32
COMPLETED


### Candidate 2: Qwen3.5-9B BF16 (text-only serving)

Run the cell below once. COMPLETED means the experiment loop ended, not that all checks passed. BLOCKED means later measurements are NOT VERIFIED; inspect the candidate's state.json, server.log and warmups.json. Proceed to the next candidate even if this one is blocked.

In [11]:
print(run_candidate(MODELS[1]))

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

... (7 outputs left out)

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

qwen35_9b concurrency 1 pass 1 complete 29/29 seconds 259.41
qwen35_9b concurrency 1 pass 2 complete 29/29 seconds 259.33
qwen35_9b concurrency 4 pass 1 complete 29/29 seconds 76.88
qwen35_9b concurrency 4 pass 2 complete 29/29 seconds 77.33
COMPLETED


### Candidate 3: Qwen3-14B official FP8

Run the cell below once. COMPLETED means the experiment loop ended, not that all checks passed. BLOCKED means later measurements are NOT VERIFIED; inspect the candidate's state.json, server.log and warmups.json. Proceed to the next candidate even if this one is blocked.

In [12]:
print(run_candidate(MODELS[2]))

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/896 [00:00<?, ?B/s]

... (5 outputs left out)

vocab.json: 0.00B [00:00, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

qwen3_14b_fp8 concurrency 1 pass 1 complete 27/29 seconds 257.3
qwen3_14b_fp8 concurrency 1 pass 2 complete 27/29 seconds 256.01
qwen3_14b_fp8 concurrency 4 pass 1 complete 25/29 seconds 69.95
qwen3_14b_fp8 concurrency 4 pass 2 complete 25/29 seconds 69.9
COMPLETED


## 10. Shared reporting helpers

Tables always retain all three candidates, including not-started or blocked runs. Ratios use the full requested denominator so failures cannot improve a score by disappearing. Performance samples use completed calls only and show success coverage separately.

A pair means one case at one concurrency across all repeats. A 23/29 ratio is a case-level result, not 23 correct token strings or 23 independent businesses. Repeated fixture questions are correlated; do not treat percentages as statistical proof of superiority.

In [13]:
from collections import defaultdict
from IPython.display import display, Markdown
REPORT_ROWS=[]

def show_table(rows):
    if not rows:
        display(Markdown("_No rows._")); return
    columns=list(rows[0])
    def clean(value):
        return str(value).replace("|","/").replace("\n"," ")
    text="| "+" | ".join(columns)+" |\n| "+" | ".join("---" for _ in columns)+" |\n"
    text+="\n".join("| "+" | ".join(clean(row.get(k,"")) for k in columns)+" |" for row in rows)
    display(Markdown(text))

def state_for(profile):
    return MODEL_RUNS.get(profile["key"],dict(status="NOT STARTED",records=[],batches=[]))

def status_ratio(passed,expected,observed):
    if observed==0:
        return "NOT VERIFIED"
    if observed<expected:
        return "PARTIAL"
    return "PASS" if passed==expected else "FAIL"

def metric(profile,name,passed,expected,observed):
    status=status_ratio(passed,expected,observed)
    row=dict(candidate=profile["key"],criterion=name,passed=passed,
             expected=expected,observed=observed,status=status)
    REPORT_ROWS.append(row)
    return f"{status}: {passed}/{expected} (observed {observed})"

def comparison_row(name,fn):
    row={"criterion":name}
    for p in MODELS:
        row[p["key"]]=fn(p,state_for(p))
    return row

def records_at(state,concurrency=None):
    return [r for r in state.get("records",[]) if concurrency is None or r["concurrency"]==concurrency]

def percentile(values,p):
    if not values:return None
    values=sorted(values);i=(len(values)-1)*p/100
    lo,hi=math.floor(i),math.ceil(i)
    return values[lo]+(values[hi]-values[lo])*(i-lo)

def classify_stability(state,selected_cases,field=None,cross=False):
    expected=len(selected_cases)*(1 if cross else len(CONCURRENCIES))
    matched=changed=unavailable=0
    for case in selected_cases:
        for concurrency in ([None] if cross else CONCURRENCIES):
            group=[r for r in records_at(state,concurrency) if r["case_id"]==case["case_id"]]
            need=REPEATS*(len(CONCURRENCIES) if cross else 1)
            keys=[(r["concurrency"],r["repeat"]) for r in group]
            valid=(len(group)==need and len(set(keys))==need and all(completed(r) for r in group))
            if field:
                valid=valid and all(r.get(field) is not None for r in group)
            if not valid:
                unavailable+=1;continue
            values=[r[field] if field else (r["content"],r["reasoning"]) for r in group]
            if all(v==values[0] for v in values[1:]):matched+=1
            else:changed+=1
    status=("NOT VERIFIED" if matched+changed==0 else
            "FAIL" if changed else "PARTIAL" if unavailable else "PASS")
    return dict(status=status,matched=matched,expected=expected,changed=changed,unavailable=unavailable)

def stability_text(profile,state,field=None,selected=None,cross=False,label=""):
    result=classify_stability(state,cases if selected is None else selected,field,cross)
    REPORT_ROWS.append(dict(candidate=profile["key"],criterion=label,**result))
    return (f'{result["status"]}: {result["matched"]}/{result["expected"]}; '
            f'changed={result["changed"]}; unavailable={result["unavailable"]}')

## 11. Criterion A — eligibility, checkpoint and context fit

Parameter PASS means the recorded count is <=15B; NOT VERIFIED means no reliable count is available. FP8 count provenance differs from a BF16 header count and must be reviewed. Release/license stays NOT VERIFIED until manually reviewed evidence is supplied. Revision date alone does not establish original release eligibility.

Startup PASS means health readiness was observed. FAIL means startup was attempted and failed; NOT VERIFIED means it was never attempted. Context-fit PASS covers every common request with its output budget; FAIL means at least one exceeds it. All three models must use the same workload hash to compare ratios.

No single combined eligibility PASS is inferred from the model's name, nominal size, or repository license tag.

In [14]:
eligibility=[]
for p in MODELS:
    s=state_for(p);e=s.get("checkpoint",{});count=e.get("parameter_count")
    startup=("PASS" if s.get("startup_ok") else "FAIL" if s.get("startup_attempted") else "NOT VERIFIED")
    row=dict(candidate=p["key"],run_status=s["status"],precision=p["quantization"] or p["dtype"],
        parameters=count if count is not None else "unknown",
        parameter_status="NOT VERIFIED" if count is None else "PASS" if count<=15_000_000_000 else "FAIL",
        count_source=e.get("parameter_count_source"),
        license_release="REVIEWED" if e.get("license_reviewed") and e.get("release_evidence_url") else "NOT VERIFIED",
        context_fit="NOT VERIFIED" if "context_fit" not in s else "PASS" if s["context_fit"] else "FAIL",
        startup=startup,load_seconds=round(s["load_seconds"],2) if s.get("load_seconds") else None,
        failure=s.get("failure",""))
    eligibility.append(row)
show_table(eligibility)
save_json("eligibility.json",eligibility)

| candidate | run_status | precision | parameters | parameter_status | count_source | license_release | context_fit | startup | load_seconds | failure |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| qwen3_8b | COMPLETED | bfloat16 | 8190735360 | PASS | actual unquantized safetensors tensor shapes | NOT VERIFIED | PASS | PASS | 98.08 |  |
| qwen35_9b | COMPLETED | bfloat16 | 9653104368 | PASS | actual unquantized safetensors tensor shapes | NOT VERIFIED | PASS | PASS | 132.1 |  |
| qwen3_14b_fp8 | COMPLETED | fp8 | 14768307200 | PASS | Hub metadata of pinned unquantized base; review before release | NOT VERIFIED | PASS | PASS | 104.09 |  |

PosixPath('/content/datamate_comparison/20261109T111420_557266Z/eligibility.json')

## 12. Criterion B — transport, schema, parser and SQL validity

Every ratio has a distinct denominator:
- Complete requests: all scheduled requests, including diagnostics.
- Parser invariants: parse calls only; catches invented time spans.
- Time presence: fixture parse calls with a reference time; catches omitted time, not correct resolution.
- SQL static/type validity: non-diagnostic solve/repair calls; not answer accuracy.

PASS requires every scheduled observation to satisfy the check. FAIL with 28/29 is better coverage than FAIL 23/29 under identical conditions, but both have defects to inspect. PARTIAL means fewer records were collected; NOT VERIFIED means no relevant requests ran. Model-generated notes are never treated as proof that SQL applied a filter.

In [15]:
validity_rows=[]
def ratio_for(p,s,label,predicate,selected):
    rs=[r for r in s.get("records",[]) if selected(r)]
    if label=="Complete HTTP/schema/output":
        count=len(cases)
    elif label=="Parser time presence":
        count=sum(c["question"].question in reference_frames and bool(reference_frames[c["question"].question].time)
                  for c in parse_cases)
    elif label=="Parser invariants":
        count=len(parse_cases)
    else:
        count=len(answer_cases)
    expected=count*len(CONCURRENCIES)*REPEATS
    return metric(p,label,sum(bool(predicate(r)) for r in rs),expected,len(rs))
definitions=[
 ("Complete HTTP/schema/output",lambda r:completed(r),lambda r:True),
 ("Parser invariants",lambda r:completed(r) and r["app_valid"],lambda r:r["stage"]=="parse"),
 ("Parser time presence",lambda r:completed(r) and r.get("time_presence_ok") is True,
  lambda r:r["stage"]=="parse" and r["case_id"] in {
      c["case_id"] for c in parse_cases if c["question"].question in reference_frames
      and reference_frames[c["question"].question].time}),
 ("SQL static validity",lambda r:completed(r) and r.get("sql_ok") is True,
  lambda r:r["case_id"] in {c["case_id"] for c in answer_cases}),
 ("SQL execution + typed result",lambda r:completed(r) and r.get("typed_ok") is True,
  lambda r:r["case_id"] in {c["case_id"] for c in answer_cases}),
]
for label,predicate,selected in definitions:
    if label=="SQL execution + typed result" and not EXECUTE_FIXTURE_SQL:
        validity_rows.append(comparison_row(label,lambda p,s:"NOT VERIFIED: SQL execution disabled"))
    else:
        validity_rows.append(comparison_row(label,
            lambda p,s,label=label,predicate=predicate,selected=selected:
                ratio_for(p,s,label,predicate,selected)))
show_table(validity_rows)
save_json("validity_comparison.json",validity_rows)

| criterion | qwen3_8b | qwen35_9b | qwen3_14b_fp8 |
| --- | --- | --- | --- |
| Complete HTTP/schema/output | PASS: 116/116 (observed 116) | PASS: 116/116 (observed 116) | FAIL: 104/116 (observed 116) |
| Parser invariants | PASS: 64/64 (observed 64) | PASS: 64/64 (observed 64) | FAIL: 52/64 (observed 64) |
| Parser time presence | FAIL: 29/52 (observed 52) | FAIL: 16/52 (observed 52) | FAIL: 38/52 (observed 52) |
| SQL static validity | PASS: 48/48 (observed 48) | PASS: 48/48 (observed 48) | PASS: 48/48 (observed 48) |
| SQL execution + typed result | FAIL: 32/48 (observed 48) | FAIL: 40/48 (observed 48) | FAIL: 40/48 (observed 48) |

PosixPath('/content/datamate_comparison/20261109T111420_557266Z/validity_comparison.json')

## 13. Criterion C — repeatability at multiple levels

Raw output compares content AND reasoning, so notes, whitespace and omitted default fields can fail it. Canonical frame equality fills Pydantic defaults but does not claim semantic equivalence of different frames. SQL-text equality ignores notes/trailing semicolons; answer equality compares formatted wire answers.

For each case/concurrency, all repeats must complete before comparison. **changed** means comparable valid outputs differ; **unavailable** means missing/invalid/truncated output or no typed answer. PASS: every expected group matched. FAIL: at least one genuine difference. PARTIAL: observed groups match but some are unavailable. NOT VERIFIED: no group can be compared.

23/29 vs 28/29 means six vs one group did not match/was unavailable; inspect both counts before concluding which model is more stable. Identical wrong answers can achieve PASS here; inspect accuracy separately. Cross-concurrency counts one group per case and is diagnostic.

In [16]:
stability_rows=[]
levels=[
 ("Raw exact within concurrency",None,cases,False),
 ("Canonical parser frame within concurrency","frame_key",parse_cases,False),
 ("SQL text within concurrency","sql_key",answer_cases,False),
 ("Final fixture answer within concurrency","answer_key",answer_cases,False),
 ("Raw exact across concurrency",None,cases,True),
 ("Final fixture answer across concurrency","answer_key",answer_cases,True),
]
for label,field,selected,cross in levels:
    stability_rows.append(comparison_row(label,
        lambda p,s,label=label,field=field,selected=selected,cross=cross:
            stability_text(p,s,field,selected,cross,label)))
show_table(stability_rows)
save_json("stability_comparison.json",stability_rows)

| criterion | qwen3_8b | qwen35_9b | qwen3_14b_fp8 |
| --- | --- | --- | --- |
| Raw exact within concurrency | FAIL: 48/58; changed=10; unavailable=0 | FAIL: 51/58; changed=7; unavailable=0 | FAIL: 46/58; changed=6; unavailable=6 |
| Canonical parser frame within concurrency | FAIL: 31/32; changed=1; unavailable=0 | PASS: 32/32; changed=0; unavailable=0 | PARTIAL: 26/32; changed=0; unavailable=6 |
| SQL text within concurrency | FAIL: 21/24; changed=3; unavailable=0 | FAIL: 21/24; changed=3; unavailable=0 | FAIL: 23/24; changed=1; unavailable=0 |
| Final fixture answer within concurrency | PARTIAL: 16/24; changed=0; unavailable=8 | FAIL: 19/24; changed=1; unavailable=4 | PARTIAL: 20/24; changed=0; unavailable=4 |
| Raw exact across concurrency | FAIL: 20/29; changed=9; unavailable=0 | FAIL: 19/29; changed=10; unavailable=0 | FAIL: 15/29; changed=10; unavailable=4 |
| Final fixture answer across concurrency | PARTIAL: 8/12; changed=0; unavailable=4 | FAIL: 7/12; changed=3; unavailable=2 | PARTIAL: 10/12; changed=0; unavailable=2 |

PosixPath('/content/datamate_comparison/20261109T111420_557266Z/stability_comparison.json')

## 14. Criterion D — fixture correctness

The primary comparison fixes concurrency to the first configured value and uses repeat 0. Each case counts once; reruns are not extra independent samples. A missing/invalid/unexecutable result is incorrect for this strict functional success ratio. If the run never produced observations, the result is NOT VERIFIED; a partially interrupted run is PARTIAL.

PASS means all selected isolated solver/repair fixture answers match the hand-derived references. FAIL 28/29 would be stronger than FAIL 23/29 only on the same 29 questions. Actual denominators below may be smaller than 29 because parse/diagnostics are excluded. The fixture uses gold frames and is synthetic: it is NOT held-out PA, a full-pipeline score, or proof of generalization. There is no automatic final model winner.

Inspect all repeated attempts separately for repairs; the dedicated repair case is not automatic recovery of every solve failure.

In [17]:
PRIMARY_CONCURRENCY=CONCURRENCIES[0]
accuracy_rows=[]
for p in MODELS:
    s=state_for(p)
    selected=[r for r in records_at(s,PRIMARY_CONCURRENCY) if r["repeat"]==0
              and r["case_id"] in {c["case_id"] for c in answer_cases}]
    correct=sum(r.get("fixture_score")==1 for r in selected)
    status=status_ratio(correct,len(answer_cases),len(selected))
    accuracy_rows.append(dict(candidate=p["key"],status=status,
        correct=correct,expected=len(answer_cases),observed=len(selected),
        wrong_answer=sum(r.get("fixture_score")==-1 for r in selected),
        no_scored_answer=sum(r.get("fixture_score") is None for r in selected),
        concurrency=PRIMARY_CONCURRENCY,repeat=0))
show_table(accuracy_rows)
save_json("fixture_accuracy.json",accuracy_rows)

| candidate | status | correct | expected | observed | wrong_answer | no_scored_answer | concurrency | repeat |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| qwen3_8b | FAIL | 6 | 12 | 12 | 2 | 4 | 1 | 0 |
| qwen35_9b | FAIL | 7 | 12 | 12 | 3 | 2 | 1 | 0 |
| qwen3_14b_fp8 | FAIL | 7 | 12 | 12 | 3 | 2 | 1 | 0 |

PosixPath('/content/datamate_comparison/20261109T111420_557266Z/fixture_accuracy.json')

## 15. Criterion E — memory and performance

Memory PASS means the sampled total GPU usage never exceeded 32 GiB during an actual serving attempt; FAIL means it did. Missing samples/attempts are NOT VERIFIED. Sampling may miss spikes, and an OOM startup can fail despite a low observed peak.

Compare p50/p95 for the same stage/concurrency, alongside complete/expected requests. A fast model with many failures is not automatically better. Throughput is aggregate completion tokens divided by evaluator batch wall time, including CPU validation; it is not single-stream decode speed. Tokenizers differ, so also compare requests/s. Values are **MEASURED**, not full-run PASS.

The project parse p95<=15s and solve p95<=30s are A100 diagnostics here, not target-hardware acceptance. Prefix caching is enabled; initial warm-up and load time are excluded from benchmark latency. An 8B BF16 vs 14B FP8 speed comparison describes these deployments, not parameter count alone.

In [18]:
memory_rows=[]; performance_rows=[]
for p in MODELS:
    s=state_for(p)
    samples=[r["used_mib"] for r in s.get("gpu_samples",[]) if "used_mib" in r]
    peak=max(samples) if samples else None
    memory_rows.append(dict(candidate=p["key"],peak_mib=peak,
        status="NOT VERIFIED" if peak is None or not s.get("startup_attempted") else
               "PASS" if peak<=32*1024 else "FAIL",
        engine_budget_gib=TARGET_EXECUTOR_GIB,engine_fraction=s.get("engine_memory_fraction")))
    for concurrency in CONCURRENCIES:
        for stage in ("parse","solve","repair"):
            rs=[r for r in records_at(s,concurrency) if r["stage"]==stage]
            done=[r for r in rs if completed(r)]
            latency=[r["elapsed_s"] for r in done]
            p95=percentile(latency,95)
            expected=sum(c["stage"]==stage for c in cases)*REPEATS
            performance_rows.append(dict(candidate=p["key"],concurrency=concurrency,stage=stage,
                completed=len(done),expected=expected,
                p50_s=round(percentile(latency,50),3) if latency else None,
                p95_s=round(p95,3) if latency else None,
                timing_status="NOT VERIFIED" if not latency else "MEASURED",
                a100_project_target="NOT VERIFIED" if p95 is None else
                    "PARTIAL" if len(done)<expected else "PASS" if p95<=(15 if stage=="parse" else 30) else "FAIL"))
show_table(memory_rows);show_table(performance_rows)
batch_rows=[dict(candidate=p["key"],**b) for p in MODELS for b in state_for(p).get("batches",[])]
show_table(batch_rows)
save_json("memory.json",memory_rows)
save_json("performance.json",performance_rows)
save_json("batches.json",batch_rows)

| candidate | peak_mib | status | engine_budget_gib | engine_fraction |
| --- | --- | --- | --- | --- |
| qwen3_8b | 19758.0 | PASS | 28.8 | 0.9 |
| qwen35_9b | 19552.0 | PASS | 28.8 | 0.9 |
| qwen3_14b_fp8 | 19642.0 | PASS | 28.8 | 0.9 |

| candidate | concurrency | stage | completed | expected | p50_s | p95_s | timing_status | a100_project_target |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| qwen3_8b | 1 | parse | 32 | 32 | 4.39 | 9.413 | MEASURED | PASS |
| qwen3_8b | 1 | solve | 24 | 24 | 8.567 | 10.935 | MEASURED | PASS |
| qwen3_8b | 1 | repair | 2 | 2 | 6.745 | 6.746 | MEASURED | PASS |
| qwen3_8b | 4 | parse | 32 | 32 | 6.386 | 10.2 | MEASURED | PASS |
| qwen3_8b | 4 | solve | 24 | 24 | 9.217 | 11.542 | MEASURED | PASS |
| qwen3_8b | 4 | repair | 2 | 2 | 7.411 | 7.433 | MEASURED | PASS |
| qwen35_9b | 1 | parse | 32 | 32 | 3.416 | 8.829 | MEASURED | PASS |
| qwen35_9b | 1 | solve | 24 | 24 | 15.335 | 21.492 | MEASURED | PASS |
| qwen35_9b | 1 | repair | 2 | 2 | 8.064 | 8.065 | MEASURED | PASS |
| qwen35_9b | 4 | parse | 32 | 32 | 3.797 | 10.164 | MEASURED | PASS |
| qwen35_9b | 4 | solve | 24 | 24 | 17.677 | 24.579 | MEASURED | PASS |
| qwen35_9b | 4 | repair | 2 | 2 | 10.098 | 10.351 | MEASURED | PASS |
| qwen3_14b_fp8 | 1 | parse | 28 | 32 | 6.929 | 11.452 | MEASURED | PARTIAL |
| qwen3_14b_fp8 | 1 | solve | 24 | 24 | 9.835 | 16.506 | MEASURED | PASS |
| qwen3_14b_fp8 | 1 | repair | 2 | 2 | 8.45 | 8.458 | MEASURED | PASS |
| qwen3_14b_fp8 | 4 | parse | 24 | 32 | 6.935 | 11.077 | MEASURED | PARTIAL |
| qwen3_14b_fp8 | 4 | solve | 24 | 24 | 10.296 | 17.058 | MEASURED | PASS |
| qwen3_14b_fp8 | 4 | repair | 2 | 2 | 8.839 | 8.874 | MEASURED | PASS |

| candidate | concurrency | repeat | wall_seconds | completed | total | requests_per_second | aggregate_output_tokens_per_second |
| --- | --- | --- | --- | --- | --- | --- | --- |
| qwen3_8b | 1 | 0 | 189.43777471499993 | 29 | 29 | 0.15308456850081306 | 16.184734035292855 |
| qwen3_8b | 1 | 1 | 188.20518176300016 | 29 | 29 | 0.15408714961163306 | 16.290731058940242 |
| qwen3_8b | 4 | 0 | 54.297741489000146 | 29 | 29 | 0.5340921961896875 | 56.42960307328284 |
| qwen3_8b | 4 | 1 | 54.32170340499988 | 29 | 29 | 0.5338566020985782 | 57.840601510128714 |
| qwen35_9b | 1 | 0 | 259.41177725199987 | 29 | 29 | 0.11179137781330795 | 15.350112636296284 |
| qwen35_9b | 1 | 1 | 259.32561334599995 | 29 | 29 | 0.11182852177932512 | 15.355212887078366 |
| qwen35_9b | 4 | 0 | 76.88071662699986 | 29 | 29 | 0.3772077222003345 | 51.91158687246673 |
| qwen35_9b | 4 | 1 | 77.32841995699982 | 29 | 29 | 0.3750238271534074 | 51.35240060779933 |
| qwen3_14b_fp8 | 1 | 0 | 257.29619354399983 | 27 | 29 | 0.10493742495021703 | 14.302582363585136 |
| qwen3_14b_fp8 | 1 | 1 | 256.006316772 | 27 | 29 | 0.10546614763434249 | 14.444956072288834 |
| qwen3_14b_fp8 | 4 | 0 | 69.95188162299974 | 25 | 29 | 0.3573885279417582 | 53.33666391002799 |
| qwen3_14b_fp8 | 4 | 1 | 69.89615408100053 | 25 | 29 | 0.357673470431982 | 53.0930499509234 |

PosixPath('/content/datamate_comparison/20261109T111420_557266Z/batches.json')

## 16. Inspect failures rather than attributing them to the GPU

This cell lists actual failed case IDs and reason codes. E_TYPE_CONTRACT commonly means SQL returned the wrong columns/shape; E_SQL_EXEC can mean SQL syntax or execution errors; length means truncated generation. These are not evidence of defective hardware.

For repeatability, notes-only/default-field differences differ from changed SQL or changed answers. Review raw and semantic-level tables together. HTTP 200 + stop proves generation completed, not that business rules were followed. Identical SQL missing valid-order filters is a stable error.

In [19]:
failure_rows=[]
for p in MODELS:
    for r in state_for(p).get("records",[]):
        relevant=r["case_id"] in {c["case_id"] for c in answer_cases}
        if not completed(r) or not r["app_valid"] or (relevant and r.get("typed_ok") is not True):
            failure_rows.append(dict(candidate=p["key"],case_id=r["case_id"],
                concurrency=r["concurrency"],repeat=r["repeat"],http=r["http_status"],
                finish_reason=r["finish_reason"],error=r.get("error"),
                schema_error=r.get("validation_error"),sql_error=r.get("sql_error"),
                type_error=r.get("type_error"),sql=r.get("sql"),
                execution_error=(r.get("execution") or {}).get("error_text")))
show_table(failure_rows[:40])
print("Total failed attempts:",len(failure_rows),"; displayed first 40; all saved below.")
save_json("failures.json",failure_rows)

| candidate | case_id | concurrency | repeat | http | finish_reason | error | schema_error | sql_error | type_error | sql | execution_error |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| qwen3_8b | solve:fx_q04 | 1 | 0 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT ten_hang, SUM(so_luong) AS total_quantity, ROW_NUMBER() OVER (ORDER BY SUM(so_luong) DESC, ma_hang ASC) AS sort_key FROM ban_hang WHERE ngay_dat BETWEEN '2026-03-01' AND '2026-03-04' GROUP BY ten_hang, ma_hang HAVING SUM(so_luong) > 0 ORDER BY total_quantity DESC, ma_hang ASC LIMIT 3 | None |
| qwen3_8b | solve:fx_q05 | 1 | 0 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT   SUM(CASE WHEN ngay_dat BETWEEN '2026-01-01' AND '2026-01-31' THEN thanh_tien ELSE 0 END) AS doanh_thu_thang_1,   SUM(CASE WHEN ngay_dat BETWEEN '2026-02-01' AND '2026-02-28' THEN thanh_tien ELSE 0 END) AS doanh_thu_thang_2 FROM ban_hang; | None |
| qwen3_8b | solve:fx_q06 | 1 | 0 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT ngay_dat, SUM(thanh_tien) AS doanh_thu FROM ban_hang WHERE ngay_dat BETWEEN '2026-02-01' AND '2026-02-28' AND trang_thai NOT IN ('Huỷ', 'Trả hàng') GROUP BY ngay_dat ORDER BY doanh_thu DESC LIMIT 1; | None |
| qwen3_8b | solve:fx_q11 | 1 | 0 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT khach_hang, SUM(thanh_tien) AS doanh_thu FROM ban_hang WHERE ngay_dat BETWEEN '2026-02-01' AND '2026-02-28' AND trang_thai NOT IN ('Huỷ', 'Trả hàng') GROUP BY khach_hang ORDER BY doanh_thu DESC LIMIT 1; | None |
| qwen3_8b | solve:fx_q04 | 1 | 1 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT ten_hang, SUM(so_luong) AS total_quantity, ROW_NUMBER() OVER (ORDER BY SUM(so_luong) DESC, ma_hang ASC) AS sort_key FROM ban_hang WHERE ngay_dat BETWEEN '2026-03-01' AND '2026-03-04' GROUP BY ten_hang, ma_hang HAVING SUM(so_luong) > 0 ORDER BY total_quantity DESC, ma_hang ASC LIMIT 3 | None |
| qwen3_8b | solve:fx_q05 | 1 | 1 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT   SUM(CASE WHEN ngay_dat BETWEEN '2026-01-01' AND '2026-01-31' THEN thanh_tien ELSE 0 END) AS thang_1,   SUM(CASE WHEN ngay_dat BETWEEN '2026-02-01' AND '2026-02-28' THEN thanh_tien ELSE 0 END) AS thang_2 FROM ban_hang; | None |
| qwen3_8b | solve:fx_q06 | 1 | 1 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT ngay_dat, SUM(thanh_tien) AS doanh_thu FROM ban_hang WHERE ngay_dat BETWEEN '2026-02-01' AND '2026-02-28' AND trang_thai NOT IN ('Huỷ', 'Trả hàng') GROUP BY ngay_dat ORDER BY doanh_thu DESC LIMIT 1; | None |
| qwen3_8b | solve:fx_q11 | 1 | 1 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT khach_hang, SUM(thanh_tien) AS doanh_thu FROM ban_hang WHERE ngay_dat BETWEEN '2026-02-01' AND '2026-02-28' AND trang_thai NOT IN ('Huỷ', 'Trả hàng') GROUP BY khach_hang ORDER BY doanh_thu DESC LIMIT 1 | None |
| qwen3_8b | solve:fx_q04 | 4 | 0 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT ten_hang, SUM(so_luong) AS total_quantity, ROW_NUMBER() OVER (ORDER BY SUM(so_luong) DESC, ma_hang ASC) AS sort_key FROM ban_hang WHERE ngay_dat BETWEEN '2026-03-01' AND '2026-03-04' GROUP BY ten_hang, ma_hang HAVING SUM(so_luong) > 0 ORDER BY total_quantity DESC, ma_hang ASC LIMIT 3 | None |
| qwen3_8b | solve:fx_q05 | 4 | 0 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT   SUM(CASE WHEN ngay_dat BETWEEN '2026-01-01' AND '2026-01-31' THEN thanh_tien ELSE 0 END) AS thang_1,   SUM(CASE WHEN ngay_dat BETWEEN '2026-02-01' AND '2026-02-28' THEN thanh_tien ELSE 0 END) AS thang_2 FROM ban_hang; | None |
| qwen3_8b | solve:fx_q06 | 4 | 0 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT ngay_dat, SUM(thanh_tien) AS doanh_thu FROM ban_hang WHERE ngay_dat BETWEEN '2026-02-01' AND '2026-02-28' AND trang_thai NOT IN ('Huỷ', 'Trả hàng') GROUP BY ngay_dat ORDER BY doanh_thu DESC LIMIT 1; | None |
| qwen3_8b | solve:fx_q11 | 4 | 0 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT khach_hang, SUM(thanh_tien) AS doanh_thu FROM ban_hang WHERE ngay_dat BETWEEN '2026-02-01' AND '2026-02-28' AND trang_thai NOT IN ('Huỷ', 'Trả hàng') GROUP BY khach_hang ORDER BY doanh_thu DESC LIMIT 1; | None |
| qwen3_8b | solve:fx_q04 | 4 | 1 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT ten_hang, SUM(so_luong) AS total_quantity, ROW_NUMBER() OVER (ORDER BY SUM(so_luong) DESC, ma_hang ASC) AS sort_key FROM ban_hang WHERE ngay_dat BETWEEN '2026-03-01' AND '2026-03-04' GROUP BY ten_hang, ma_hang HAVING SUM(so_luong) > 0 ORDER BY total_quantity DESC, ma_hang ASC LIMIT 3 | None |
| qwen3_8b | solve:fx_q05 | 4 | 1 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT   SUM(CASE WHEN ngay_dat BETWEEN '2026-01-01' AND '2026-01-31' THEN thanh_tien ELSE 0 END) AS doanh_thu_thang_1,   SUM(CASE WHEN ngay_dat BETWEEN '2026-02-01' AND '2026-02-28' THEN thanh_tien ELSE 0 END) AS doanh_thu_thang_2 FROM ban_hang; | None |
| qwen3_8b | solve:fx_q06 | 4 | 1 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT ngay_dat, SUM(thanh_tien) AS doanh_thu FROM ban_hang WHERE ngay_dat BETWEEN '2026-02-01' AND '2026-02-28' AND trang_thai NOT IN ('Huỷ', 'Trả hàng') GROUP BY ngay_dat ORDER BY doanh_thu DESC LIMIT 1; | None |
| qwen3_8b | solve:fx_q11 | 4 | 1 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT khach_hang, SUM(thanh_tien) AS doanh_thu FROM ban_hang WHERE ngay_dat BETWEEN '2026-02-01' AND '2026-02-28' AND trang_thai NOT IN ('Huỷ', 'Trả hàng') GROUP BY khach_hang ORDER BY doanh_thu DESC LIMIT 1; | None |
| qwen35_9b | solve:fx_q05 | 1 | 0 | 200 | stop | None | None | None | E_TYPE_CONTRACT | WITH valid_orders AS (     SELECT          ma_don,         ngay_dat,         thanh_tien     FROM ban_hang     WHERE trang_thai IN ('Hoàn thành', 'Đang giao')       AND ngay_dat <= '2026-03-04' ), revenue_jan AS (     SELECT SUM(thanh_tien) as total     FROM valid_orders     WHERE ngay_dat >= '2026-01-01' AND ngay_dat <= '2026-01-31' ), revenue_feb AS (     SELECT SUM(thanh_tien) as total     FROM valid_orders     WHERE ngay_dat >= '2026-02-01' AND ngay_dat <= '2026-02-28' ) SELECT      'tháng 1/2026' as item,     COALESCE(rj.total, 0) as value FROM revenue_jan rj UNION ALL SELECT      'tháng 2/2026' as item,     COALESCE(rf.total, 0) as value FROM revenue_feb rf ORDER BY item; | None |
| qwen35_9b | solve:fx_q06 | 1 | 0 | 200 | stop | None | None | None | E_SQL_EXEC | WITH daily_revenue AS (     SELECT          ngay_dat,         SUM(CASE WHEN trang_thai IN ('Hoàn thành', 'Đang giao') THEN thanh_tien ELSE 0 END) AS revenue     FROM ban_hang     WHERE ngay_dat >= DATE '2026-02-01'        AND ngay_dat <= DATE '2026-02-28'     GROUP BY ngay_dat ) SELECT      MAX(revenue) AS max_rev,     MAX(CASE WHEN revenue = MAX(revenue) THEN ngay_dat END) AS max_rev_date FROM daily_revenue | BinderException: Binder Error: aggregate function calls cannot be nested  LINE 12:     MAX(CASE WHEN revenue = MAX(revenue) THEN ngay_dat END) AS max_rev_date                                      ^ |
| qwen35_9b | solve:fx_q05 | 1 | 1 | 200 | stop | None | None | None | E_TYPE_CONTRACT | WITH valid_orders AS (     SELECT          ma_don,         ngay_dat,         thanh_tien     FROM ban_hang     WHERE trang_thai IN ('Hoàn thành', 'Đang giao')       AND ngay_dat <= '2026-03-04' ), revenue_jan AS (     SELECT SUM(thanh_tien) as total     FROM valid_orders     WHERE ngay_dat >= '2026-01-01' AND ngay_dat <= '2026-01-31' ), revenue_feb AS (     SELECT SUM(thanh_tien) as total     FROM valid_orders     WHERE ngay_dat >= '2026-02-01' AND ngay_dat <= '2026-02-28' ) SELECT      'tháng 1/2026' as item,     COALESCE(rj.total, 0) as value FROM revenue_jan rj UNION ALL SELECT      'tháng 2/2026' as item,     COALESCE(rf.total, 0) as value FROM revenue_feb rf ORDER BY item; | None |
| qwen35_9b | solve:fx_q06 | 1 | 1 | 200 | stop | None | None | None | E_SQL_EXEC | WITH daily_revenue AS (     SELECT          ngay_dat,         SUM(CASE WHEN trang_thai IN ('Hoàn thành', 'Đang giao') THEN thanh_tien ELSE 0 END) AS revenue     FROM ban_hang     WHERE ngay_dat >= DATE '2026-02-01'        AND ngay_dat <= DATE '2026-02-28'     GROUP BY ngay_dat ) SELECT      MAX(revenue) AS max_rev,     MAX(CASE WHEN revenue = MAX(revenue) THEN ngay_dat END) AS max_rev_date FROM daily_revenue | BinderException: Binder Error: aggregate function calls cannot be nested  LINE 12:     MAX(CASE WHEN revenue = MAX(revenue) THEN ngay_dat END) AS max_rev_date                                      ^ |
| qwen35_9b | solve:fx_q06 | 4 | 0 | 200 | stop | None | None | None | E_SQL_EXEC | WITH daily_revenue AS (     SELECT          ngay_dat,         SUM(CASE WHEN trang_thai IN ('Hoàn thành', 'Đang giao') THEN thanh_tien ELSE 0 END) AS revenue     FROM ban_hang     WHERE ngay_dat >= DATE '2026-02-01'        AND ngay_dat <= DATE '2026-02-28'     GROUP BY ngay_dat ) SELECT      MAX(revenue) AS max_rev,     MAX(CASE WHEN revenue = MAX(revenue) THEN ngay_dat END) AS max_rev_date FROM daily_revenue | BinderException: Binder Error: aggregate function calls cannot be nested  LINE 12:     MAX(CASE WHEN revenue = MAX(revenue) THEN ngay_dat END) AS max_rev_date                                      ^ |
| qwen35_9b | solve:fx_q05 | 4 | 0 | 200 | stop | None | None | None | E_TYPE_CONTRACT | WITH valid_orders AS (     SELECT          ma_don,         ngay_dat,         thanh_tien     FROM ban_hang     WHERE trang_thai IN ('Hoàn thành', 'Đang giao')       AND ngay_dat <= '2026-03-04' ), revenue_jan AS (     SELECT SUM(thanh_tien) as total     FROM valid_orders     WHERE ngay_dat >= '2026-01-01' AND ngay_dat <= '2026-01-31' ), revenue_feb AS (     SELECT SUM(thanh_tien) as total     FROM valid_orders     WHERE ngay_dat >= '2026-02-01' AND ngay_dat <= '2026-02-28' ) SELECT      'tháng 1/2026' as item,     COALESCE(rj.total, 0) as value FROM revenue_jan rj UNION ALL SELECT      'tháng 2/2026' as item,     COALESCE(rf.total, 0) as value FROM revenue_feb rf ORDER BY item; | None |
| qwen35_9b | solve:fx_q06 | 4 | 1 | 200 | stop | None | None | None | E_SQL_EXEC | WITH daily_revenue AS (     SELECT          ngay_dat,         SUM(CASE WHEN trang_thai IN ('Hoàn thành', 'Đang giao') THEN thanh_tien ELSE 0 END) AS revenue     FROM ban_hang     WHERE ngay_dat >= DATE '2026-02-01'        AND ngay_dat <= DATE '2026-02-28'     GROUP BY ngay_dat ) SELECT      MAX(revenue) AS max_rev,     MAX(CASE WHEN revenue = MAX(revenue) THEN ngay_dat END) AS max_rev_date FROM daily_revenue ORDER BY max_rev DESC, max_rev_date ASC LIMIT 1; | BinderException: Binder Error: aggregate function calls cannot be nested  LINE 12:     MAX(CASE WHEN revenue = MAX(revenue) THEN ngay_dat END) AS max_rev_date                                      ^ |
| qwen35_9b | solve:fx_q05 | 4 | 1 | 200 | stop | None | None | None | E_TYPE_CONTRACT | WITH valid_orders AS (     SELECT          ma_don,         ngay_dat,         thanh_tien     FROM ban_hang     WHERE trang_thai IN ('Hoàn thành', 'Đang giao')       AND ngay_dat <= '2026-03-04' ), revenue_jan AS (     SELECT SUM(thanh_tien) as total     FROM valid_orders     WHERE ngay_dat >= '2026-01-01' AND ngay_dat <= '2026-01-31' ), revenue_feb AS (     SELECT SUM(thanh_tien) as total     FROM valid_orders     WHERE ngay_dat >= '2026-02-01' AND ngay_dat <= '2026-02-28' ) SELECT      '2026-01' as period,     COALESCE(rj.total, 0) as value FROM revenue_jan rj UNION ALL SELECT      '2026-02' as period,     COALESCE(rf.total, 0) as value FROM revenue_feb rf ORDER BY period; | None |
| qwen3_14b_fp8 | parse:fx_q04 | 1 | 0 | 200 | stop | None | ValidationError: 1 validation error for QuestionFrame   Value error, date_granularity is required iff expected_type is date [type=value_error, input_value={'expected_type': 'list',...'status': 'answerable'}}, input_type=dict]     For further information visit https://errors.pydantic.dev/2.11/v/value_error | None | None | None | None |
| qwen3_14b_fp8 | solve:fx_q04 | 1 | 0 | 200 | stop | None | None | None | E_SQL_EXEC | WITH sales AS (SELECT ma_hang, SUM(so_luong) AS total_sold FROM ban_hang WHERE ngay_dat BETWEEN '2026-03-01' AND '2026-03-04' AND trang_thai NOT IN ('Huỷ', 'Trả hàng') GROUP BY ma_hang), ranked_sales AS (SELECT *, ROW_NUMBER() OVER (ORDER BY total_sold DESC, ma_hang ASC) AS rank FROM sales) SELECT ten_hang AS item, total_sold AS sort_key, ma_hang AS tie_code FROM ranked_sales WHERE rank <= 3 ORDER BY sort_key DESC, tie_code ASC | BinderException: Binder Error: Referenced column "ten_hang" not found in FROM clause! Candidate bindings: "ma_hang"  LINE 1: ... BY total_sold DESC, ma_hang ASC) AS rank FROM sales) SELECT ten_hang AS item, total_sold AS sort_key, ma_hang AS tie_code...                                                                         ^ |
| qwen3_14b_fp8 | solve:fx_q05 | 1 | 0 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT SUM(CASE WHEN ngay_dat BETWEEN '2026-01-01' AND '2026-01-31' THEN thanh_tien ELSE 0 END) AS doanh_thu_thang_1, SUM(CASE WHEN ngay_dat BETWEEN '2026-02-01' AND '2026-02-28' THEN thanh_tien ELSE 0 END) AS doanh_thu_thang_2 FROM ban_hang WHERE trang_thai NOT IN ('Huỷ', 'Trả hàng') AND ngay_dat <= '2026-03-04' | None |
| qwen3_14b_fp8 | parse:fx_q13 | 1 | 0 | 200 | stop | None | ValidationError: 1 validation error for QuestionFrame   Value error, date_granularity is required iff expected_type is date [type=value_error, input_value={'expected_type': 'table'...'status': 'answerable'}}, input_type=dict]     For further information visit https://errors.pydantic.dev/2.11/v/value_error | None | None | None | None |
| qwen3_14b_fp8 | parse:fx_q04 | 1 | 1 | 200 | stop | None | ValidationError: 1 validation error for QuestionFrame   Value error, date_granularity is required iff expected_type is date [type=value_error, input_value={'expected_type': 'list',...'status': 'answerable'}}, input_type=dict]     For further information visit https://errors.pydantic.dev/2.11/v/value_error | None | None | None | None |
| qwen3_14b_fp8 | solve:fx_q04 | 1 | 1 | 200 | stop | None | None | None | E_SQL_EXEC | WITH sales AS (SELECT ma_hang, SUM(so_luong) AS total_sold FROM ban_hang WHERE ngay_dat BETWEEN '2026-03-01' AND '2026-03-04' AND trang_thai NOT IN ('Huỷ', 'Trả hàng') GROUP BY ma_hang), ranked_sales AS (SELECT *, ROW_NUMBER() OVER (ORDER BY total_sold DESC, ma_hang ASC) AS rank FROM sales) SELECT ten_hang AS item, total_sold AS sort_key, ma_hang AS tie_code FROM ranked_sales WHERE rank <= 3 ORDER BY sort_key DESC, tie_code ASC | BinderException: Binder Error: Referenced column "ten_hang" not found in FROM clause! Candidate bindings: "ma_hang"  LINE 1: ... BY total_sold DESC, ma_hang ASC) AS rank FROM sales) SELECT ten_hang AS item, total_sold AS sort_key, ma_hang AS tie_code...                                                                         ^ |
| qwen3_14b_fp8 | solve:fx_q05 | 1 | 1 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT SUM(CASE WHEN ngay_dat BETWEEN '2026-01-01' AND '2026-01-31' THEN thanh_tien ELSE 0 END) AS doanh_thu_thang_1, SUM(CASE WHEN ngay_dat BETWEEN '2026-02-01' AND '2026-02-28' THEN thanh_tien ELSE 0 END) AS doanh_thu_thang_2 FROM ban_hang WHERE trang_thai NOT IN ('Huỷ', 'Trả hàng') AND ngay_dat <= '2026-03-04' | None |
| qwen3_14b_fp8 | parse:fx_q13 | 1 | 1 | 200 | stop | None | ValidationError: 1 validation error for QuestionFrame   Value error, date_granularity is required iff expected_type is date [type=value_error, input_value={'expected_type': 'table'...'status': 'answerable'}}, input_type=dict]     For further information visit https://errors.pydantic.dev/2.11/v/value_error | None | None | None | None |
| qwen3_14b_fp8 | parse:fx_q03 | 4 | 0 | 200 | stop | None | ValidationError: 1 validation error for QuestionFrame   Value error, date_granularity is required iff expected_type is date [type=value_error, input_value={'expected_type': 'percen...'status': 'answerable'}}, input_type=dict]     For further information visit https://errors.pydantic.dev/2.11/v/value_error | None | None | None | None |
| qwen3_14b_fp8 | parse:fx_q04 | 4 | 0 | 200 | stop | None | ValidationError: 1 validation error for QuestionFrame   Value error, date_granularity is required iff expected_type is date [type=value_error, input_value={'expected_type': 'list',...'status': 'answerable'}}, input_type=dict]     For further information visit https://errors.pydantic.dev/2.11/v/value_error | None | None | None | None |
| qwen3_14b_fp8 | solve:fx_q04 | 4 | 0 | 200 | stop | None | None | None | E_SQL_EXEC | WITH sales AS (SELECT ma_hang, SUM(so_luong) AS total_sold FROM ban_hang WHERE ngay_dat BETWEEN '2026-03-01' AND '2026-03-04' AND trang_thai NOT IN ('Huỷ', 'Trả hàng') GROUP BY ma_hang), ranked_sales AS (SELECT *, ROW_NUMBER() OVER (ORDER BY total_sold DESC, ma_hang ASC) AS rank FROM sales) SELECT ten_hang AS item, total_sold AS sort_key, ma_hang AS tie_code FROM ranked_sales WHERE rank <= 3 ORDER BY sort_key DESC, tie_code ASC | BinderException: Binder Error: Referenced column "ten_hang" not found in FROM clause! Candidate bindings: "ma_hang"  LINE 1: ... BY total_sold DESC, ma_hang ASC) AS rank FROM sales) SELECT ten_hang AS item, total_sold AS sort_key, ma_hang AS tie_code...                                                                         ^ |
| qwen3_14b_fp8 | solve:fx_q05 | 4 | 0 | 200 | stop | None | None | None | E_TYPE_CONTRACT | SELECT SUM(CASE WHEN ngay_dat BETWEEN '2026-01-01' AND '2026-01-31' THEN thanh_tien ELSE 0 END) AS doanh_thu_thang_1, SUM(CASE WHEN ngay_dat BETWEEN '2026-02-01' AND '2026-02-28' THEN thanh_tien ELSE 0 END) AS doanh_thu_thang_2 FROM ban_hang WHERE trang_thai NOT IN ('Huỷ', 'Trả hàng') AND ngay_dat <= '2026-03-04' | None |
| qwen3_14b_fp8 | parse:fx_q13 | 4 | 0 | 200 | stop | None | ValidationError: 1 validation error for QuestionFrame   Value error, date_granularity is required iff expected_type is date [type=value_error, input_value={'expected_type': 'table'...'status': 'answerable'}}, input_type=dict]     For further information visit https://errors.pydantic.dev/2.11/v/value_error | None | None | None | None |
| qwen3_14b_fp8 | parse:temporal_0 | 4 | 0 | 200 | stop | None | ValidationError: 1 validation error for QuestionFrame   Value error, date_granularity is required iff expected_type is date [type=value_error, input_value={'expected_type': 'boolea...'status': 'answerable'}}, input_type=dict]     For further information visit https://errors.pydantic.dev/2.11/v/value_error | None | None | None | None |
| qwen3_14b_fp8 | parse:fx_q03 | 4 | 1 | 200 | stop | None | ValidationError: 1 validation error for QuestionFrame   Value error, date_granularity is required iff expected_type is date [type=value_error, input_value={'expected_type': 'percen...'status': 'answerable'}}, input_type=dict]     For further information visit https://errors.pydantic.dev/2.11/v/value_error | None | None | None | None |
| qwen3_14b_fp8 | parse:fx_q04 | 4 | 1 | 200 | stop | None | ValidationError: 1 validation error for QuestionFrame   Value error, date_granularity is required iff expected_type is date [type=value_error, input_value={'expected_type': 'list',...'status': 'answerable'}}, input_type=dict]     For further information visit https://errors.pydantic.dev/2.11/v/value_error | None | None | None | None |

Total failed attempts: 44 ; displayed first 40; all saved below.


PosixPath('/content/datamate_comparison/20261109T111420_557266Z/failures.json')

## 17. Three-model comparison, export and conclusion template

This consolidates all three candidates without declaring an automatic winner. Missing measurements remain visible. A candidate can have better schema/SQL ratios yet worse final answers; prioritize fixture answer correctness only as a smoke signal, then run business-held-out PA through the Lead's harness.

Comparisons are between configured model+precision combinations. The same GPU/software/workload hash is required. sm_120, Docker, six-hour runtime, full training/held-out PA and final selection remain NOT VERIFIED for every A100-only run. Licensing and weight eligibility require explicit review.

Export CSV/JSON and raw evidence before leaving Colab. No model weights are put in the ZIP. CONCLUSION.md has blank fields for Lead/M4 review and follow-up M3-02/M3-08 work.

In [20]:
summary=[]
for p in MODELS:
    key=p["key"];s=state_for(p)
    accuracy=next(r for r in accuracy_rows if r["candidate"]==key)
    memory=next(r for r in memory_rows if r["candidate"]==key)
    raw=classify_stability(s,cases)
    final=classify_stability(s,answer_cases,"answer_key")
    summary.append(dict(candidate=key,checkpoint=p["model_id"],
        precision=p["quantization"] or p["dtype"],run_status=s["status"],
        complete_requests=f'{sum(completed(r) for r in s.get("records",[]))}/{expected_requests}',
        raw_repeatability=f'{raw["status"]} {raw["matched"]}/{raw["expected"]}',
        answer_repeatability=f'{final["status"]} {final["matched"]}/{final["expected"]}',
        fixture_correct=f'{accuracy["status"]} {accuracy["correct"]}/{accuracy["expected"]}',
        peak_mib=memory["peak_mib"],sm120="NOT VERIFIED",heldout_PA="NOT VERIFIED"))
show_table(summary)
def export_csv(name,rows):
    if not rows:return
    fields=list(dict.fromkeys(k for row in rows for k in row))
    with (RUN_DIR/name).open("w",newline="",encoding="utf-8-sig") as fh:
        writer=csv.DictWriter(fh,fieldnames=fields);writer.writeheader();writer.writerows(rows)
for name,rows in [
    ("model_comparison.csv",summary),("eligibility.csv",eligibility),
    ("validity_comparison.csv",validity_rows),("stability_comparison.csv",stability_rows),
    ("fixture_accuracy.csv",accuracy_rows),("memory.csv",memory_rows),
    ("performance.csv",performance_rows),("batches.csv",batch_rows),("failures.csv",failure_rows)]:
    export_csv(name,rows)
save_json("model_comparison.json",summary)
save_json("metric_counts.json",REPORT_ROWS)
save_json("experiment_manifest.json",dict(repo_commit=REPO_COMMIT,
    workload_sha256=WORKLOAD_SHA256,contracts_version=CONTRACTS_VERSION,
    templates=template_sha256(),schema_adapter=SCHEMA_ADAPTER_POLICY,
    gpu=GPUS,serving_environment=SERVER_ENV,models=MODELS,
    tested_concurrencies=CONCURRENCIES,repeats=REPEATS,max_model_len=MAX_MODEL_LEN,
    target_executor_gib=TARGET_EXECUTOR_GIB,
    warning="Synthetic gold-frame solver fixture; not full-pipeline or held-out PA."))
conclusion="""# Three-model review (fill after inspecting evidence)

- Experiment directory: {run}
- Source commit: {commit}
- Workload SHA-256: {workload}
- Models successfully served: [fill]
- Blocked models and exact causes: [fill]
- Best fixture correctness and errors still present: [fill]
- Raw vs canonical-frame / SQL / final-answer stability: [fill]
- Time-expression omissions and missing business-rule filters: [fill]
- Memory margin, latency and throughput at equal workload: [fill]
- Precision effect (14B FP8 vs 8B/9B BF16): [fill]
- License, original release and parameter-count review: [fill]
- Recommended candidates for real held-out bake-off: [fill]
- Recommended configuration for M3-02/M4-03: [fill]
- Lead/M4 sign-off: [fill]
- sm_120 / RTX 5090 / Docker: NOT VERIFIED by this A100 experiment.
- Full training / held-out PA / official runtime / final model choice: NOT VERIFIED.
"""
(RUN_DIR/"CONCLUSION.md").write_text(conclusion.format(run=RUN_DIR,commit=REPO_COMMIT,
    workload=WORKLOAD_SHA256),encoding="utf-8")
print("Artifacts:",RUN_DIR)

| candidate | checkpoint | precision | run_status | complete_requests | raw_repeatability | answer_repeatability | fixture_correct | peak_mib | sm120 | heldout_PA |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| qwen3_8b | Qwen/Qwen3-8B | bfloat16 | COMPLETED | 116/116 | FAIL 48/58 | PARTIAL 16/24 | FAIL 6/12 | 19758.0 | NOT VERIFIED | NOT VERIFIED |
| qwen35_9b | Qwen/Qwen3.5-9B | bfloat16 | COMPLETED | 116/116 | FAIL 51/58 | FAIL 19/24 | FAIL 7/12 | 19552.0 | NOT VERIFIED | NOT VERIFIED |
| qwen3_14b_fp8 | Qwen/Qwen3-14B-FP8 | fp8 | COMPLETED | 104/116 | FAIL 46/58 | PARTIAL 20/24 | FAIL 7/12 | 19642.0 | NOT VERIFIED | NOT VERIFIED |

Artifacts: /content/datamate_comparison/20261109T111420_557266Z


## 18. Download evidence (optional)

The ZIP includes comparisons, failed requests, exact responses, source/schema/model fingerprints and logs. It excludes checkpoint weights and temporary DuckDB files. Review before sharing. A failed candidate's evidence is retained alongside successful candidates.

In [21]:
import zipfile
archive_path = EVAL_ROOT / f"{RUN_STAMP}.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file() and "work" not in path.relative_to(RUN_DIR).parts:
            archive.write(path, arcname=str(path.relative_to(RUN_DIR)))
print(archive_path)
try:
    from google.colab import files
except ImportError:
    print("Outside Colab: download the archive using your file browser.")
else:
    files.download(str(archive_path))

/content/datamate_comparison/20261109T111420_557266Z.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Official sources and next steps

- [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B)
- [Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B)
- [Qwen3-14B-FP8](https://huggingface.co/Qwen/Qwen3-14B-FP8)
- [Qwen3-14B unquantized base](https://huggingface.co/Qwen/Qwen3-14B)
- [vLLM 0.17.0 supported models](https://docs.vllm.ai/en/v0.17.0/models/supported_models/)
- [FP8 / Ampere W8A16 caveats](https://docs.vllm.ai/en/v0.17.0/features/quantization/fp8/)
- [Qwen3.5 serving recipe](https://docs.vllm.ai/projects/recipes/en/latest/Qwen/Qwen3.5.html)
- [vLLM reproducibility](https://docs.vllm.ai/en/v0.17.0/usage/reproducibility/)

Qwen3.5 uses a newer architecture; common vLLM support does not guarantee every CUDA kernel works on every A100 stack. A failure is captured, not silently replaced by a different model/precision. Do not increase target VRAM to turn an OOM into a false target-fit PASS.

No benchmark was executed while authoring this notebook. Fill conclusions from your real run. Continue M3-02/replay/integration once a useful serving baseline exists; investigate prompt quality with versioned experiments and held-out PA later.